# LLM.int8() 量化

论文：[LLM.int8(): 8-bit Matrix Multiplication for Transformers at Scale](https://arxiv.org/abs/2208.07339)

**核心思想**：大部分特征走 INT8 矩阵乘法，少数异常激活特征走 FP16，避免异常值放大量化误差。

**算法流程**（`X:[M,K]`、`W:[K,N]`）：

1. 找出激活 `X` 中存在 `abs(x)>threshold` 的**列**（异常特征维度）。
2. 将 `X` 按列、`W` 按对应**行**拆分：
   - 正常路径：`X` 按行、`W` 按列进行 Vector-wise INT8 量化，整数乘法、INT32 累加、反缩放。
   - 异常路径：对应子矩阵保留 FP16，进行 FP16 矩阵乘法。
3. 将两条路径的输出相加。

**关键优势**：减少少数异常激活值对普通 INT8 量化精度的破坏；论文在大模型上报告了接近无损的表现，并非对所有输入的保证。

**阈值**：`6.0` 是论文及常见 bitsandbytes 配置中的经验默认值，不是硬编码的数学常数。

> **实现边界**：这是 CPU 数值教学 Demo。INT8 路径使用 PyTorch INT32 矩阵乘法模拟整数累加，并非优化的 INT8 GEMM 内核；每次调用重新量化权重，不代表生产级推理速度、模型精度或内存表现。


In [ ]:
import torch

def llm_int8(X, W, threshold=6.0):
    """CPU 教学版：X [M,K]，W [K,N]。"""
    # 1. 找出 X 的异常特征列
    outlier_mask = (X.abs() > threshold).any(dim=0)

    # 2. 正常路径：X 按行、W 按列量化为 INT8
    X_normal = X[:, ~outlier_mask]
    W_normal = W[~outlier_mask, :]
    if X_normal.shape[1] > 0:
        sx = X_normal.abs().amax(dim=1, keepdim=True).clamp_min(1e-12) / 127
        sw = W_normal.abs().amax(dim=0, keepdim=True).clamp_min(1e-12) / 127
        qx = torch.round(X_normal / sx).clamp(-127, 127).to(torch.int8)
        qw = torch.round(W_normal / sw).clamp(-127, 127).to(torch.int8)
        # INT32 累加演示，不是优化 INT8 GEMM
        C_int8 = (qx.to(torch.int32) @ qw.to(torch.int32)).float() * (sx * sw)
    else:
        C_int8 = torch.zeros((X.shape[0], W.shape[1]), dtype=torch.float32)

    # 3. 异常路径：X 的异常列，对应 W 的异常行，使用 FP16
    if outlier_mask.any():
        C_fp16 = (X[:, outlier_mask].half() @ W[outlier_mask, :].half()).float()
    else:
        C_fp16 = torch.zeros_like(C_int8)

    return C_int8 + C_fp16, outlier_mask


In [ ]:
# Demo：和 FP32 基线、纯 Vector-wise INT8 比较
torch.manual_seed(42)
M, K, N = 128, 256, 128
X = torch.randn(M, K) * 0.4
W = torch.randn(K, N) * 0.4
X[:, [7, 39, 120, 201]] = torch.tensor([12., -16., 20., -25.])

C_ref = X @ W
C_llm, mask = llm_int8(X, W, threshold=6.0)
C_plain, _ = llm_int8(X, W, threshold=100.0)

def mae(pred):
    return (pred - C_ref).abs().mean().item()

print("异常特征维度:", torch.where(mask)[0].tolist())
print(f"纯 Vector-wise INT8 MAE: {mae(C_plain):.6f}")
print(f"LLM.int8() MAE: {mae(C_llm):.6f}")
print(f"FP16 特征维度比例: {mask.float().mean().item():.2%}")

# 调整阈值，观察精度与异常维度数量的变化
for t in [3.0, 6.0, 10.0, 20.0, 100.0]:
    output, selected = llm_int8(X, W, threshold=t)
    print(f"threshold={t:>5.1f} | outliers={int(selected.sum()):>3} | MAE={mae(output):.6f}")
